# Multi-Agent Supply Chain Disruption — Shared Epistemic Memory POC

## Goal
Build a minimal proof-of-concept of the **Shared Epistemic Memory (SEM)** pattern from `REQUIREMENTS.md`:
three agents (Monitoring, Logistics, Customer Notification) collaborate by reading/writing a single
shared memory store — no direct agent-to-agent messaging.

## Stack
- **LLM**: NVIDIA API → `openai/gpt-oss-120b` via `langchain-nvidia-ai-endpoints`
- **Agent harness**: LangChain `create_agent` (built on LangGraph)
- **Tracing**: MLflow with `mlflow.langchain.autolog()` — every LLM call, tool call, and agent step is captured
- **Memory**: in-memory Python `dict` (skipping Redis for the POC)
- **Schemas**: Pydantic v2

## What "done" looks like
1. Three agents collaborate via SEM (in-memory dict)
2. Each agent's reasoning + tool calls visible in MLflow UI at `http://localhost:5909`
3. Stale entries flagged via TTL
4. Concurrent writes visible as last-write-wins

## Prerequisites
- `NVIDIA_API_KEY` in env
- MLflow server running: `mlflow server --host 0.0.0.0 --port 5909`


In [1]:
## Cell 1 — Imports + Environment Setup

# We import everything we need and set up MLflow tracing. The key call is
# `mlflow.langchain.autolog()` — once enabled, **every** LangChain operation
# (LLM call, tool call, agent step) is automatically captured as a trace span.
# No decorators needed.

# We also point MLflow at a local tracking server. If you don't have one running,
# MLflow will fall back to a local `mlruns/` directory.


In [1]:
import os
import mlflow
from langchain.chat_models import init_chat_model

# --- NVIDIA API config ---
API_KEY = os.environ["NVIDIA_API_KEY"]
BASE_URL = "https://integrate.api.nvidia.com/v1"
CHAT_MODEL_NAME = "openai/gpt-oss-120b"

# --- MLflow tracing config ---
TRACKING_URI = "http://localhost:5909"
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("supply-chain-disruption-poc")

# Auto-trace ALL LangChain calls (LLM, tools, agents, chains)
mlflow.langchain.autolog()

print("✓ MLflow tracing enabled")
print(f"  Tracking URI: {TRACKING_URI}")
print(f"  Experiment:   supply-chain-disruption-poc")

✓ MLflow tracing enabled
  Tracking URI: http://localhost:5909
  Experiment:   supply-chain-disruption-poc


## Cell 2 — Initialize the LLM + Smoke Test

We use LangChain's `init_chat_model` factory — it picks the right client based
on `model_provider="nvidia"`. The smoke test confirms credentials work before
we build anything on top.


In [2]:
llm = init_chat_model(
    CHAT_MODEL_NAME,
    model_provider="nvidia",
    base_url=BASE_URL,
    api_key=API_KEY,
)

# Smoke test — confirms the model is reachable
response = llm.invoke("Reply with just the word 'pong'.")
print(f"Smoke test: {response.content}")


Smoke test: pong


## Cell 3 — Pydantic Schemas for Memory Entries

Per the FDS, every entry in the shared memory must be typed. We define two
schemas:

- **`ShipmentStatus`** — current state of a shipment (status, reason, source agent, TTL)
- **`EventLog`** — discrete events that happen in the supply chain (storm detected, reroute planned, customer notified)

Each entry tracks:
- `source_agent_id` — who wrote it (for trust/audit)
- `timestamp` — when it was written
- `ttl_seconds` — how long it's valid before becoming "stale"
- `version` — for optimistic locking (incremented on every update)


In [3]:
from pydantic import BaseModel, Field
from typing import Literal
import time


class ShipmentStatus(BaseModel):
    """Current state of a shipment in the supply chain."""
    shipment_id: str
    status: Literal["in_transit", "delayed", "rerouted", "delivered", "lost"]
    reason: str = ""
    source_agent_id: str
    timestamp: float = Field(default_factory=time.time)
    ttl_seconds: int = 3600  # 1 hour default
    version: int = 1


class EventLog(BaseModel):
    """A discrete event in the supply chain."""
    event_id: str
    event_type: Literal["disruption_detected", "reroute_planned", "customer_notified", "delivery_confirmed"]
    shipment_id: str
    details: str = ""
    source_agent_id: str
    timestamp: float = Field(default_factory=time.time)
    ttl_seconds: int = 86400  # 24 hours default
    version: int = 1


print("✓ Schemas defined: ShipmentStatus, EventLog")


✓ Schemas defined: ShipmentStatus, EventLog


## Cell 4 — SharedEpistemicMemory (Upstash Redis Backend)

This is the heart of the POC. The store is **Upstash Redis** — a serverless,
HTTP-based Redis that works from anywhere (no TCP needed). Every entry is
serialized as JSON and stored under a key.

**Key operations:**
- `write(key, entry)` — `SET key value` (last-write-wins for POC)
- `read(key)` — `GET key`, returns `None` if missing or stale
- `update(key, entry)` — `SET` with optimistic locking via `version`
- `delete(key)` — `DEL key`
- `list_keys()` — `KEYS *` (for agents to discover state)
- `is_fresh(entry)` — check if entry is past its TTL

**Why Upstash?** It's the closest match to the FDS's Redis requirement
without needing a local Redis server. The HTTP-based SDK works in any
environment, and we already have the credentials in `.zshrc`.


In [5]:
from dotenv import load_dotenv
from upstash_redis import Redis

load_dotenv()

# Connect to Upstash Redis using env vars (UPSTASH_REDIS_REST_URL, UPSTASH_REDIS_REST_TOKEN)
redis_client = Redis(url=os.environ["UPSTASH_REDIS_REST_URL"], token=os.environ["UPSTASH_REDIS_REST_TOKEN"])

In [6]:
class SharedEpistemicMemory:
    """Upstash Redis-backed implementation of the Shared Epistemic Memory pattern.

    All agents read/write through this single object. The store is the
    authoritative source of truth — agents never communicate directly.

    Entries are serialized as JSON. Each entry carries its own `timestamp`,
    `ttl_seconds`, and `version` for staleness tracking and optimistic locking.
    """

    def __init__(self, client: Redis):
        self._client = client

    def write(self, key: str, entry: BaseModel) -> None:
        """Create or overwrite an entry. Last-write-wins for POC."""
        self._client.set(key, entry.model_dump_json())

    def read(self, key: str) -> BaseModel | None:
        """Read an entry. Returns None if missing or stale."""
        raw = self._client.get(key)
        if raw is None:
            return None
        # Upstash returns bytes on some responses — normalize to str
        if isinstance(raw, bytes):
            raw = raw.decode("utf-8")
        entry = self._deserialize(key, raw)
        if entry is None:
            return None
        if not self.is_fresh(entry):
            return None  # stale — caller should treat as missing
        return entry

    def update(self, key: str, entry: BaseModel, expected_version: int) -> BaseModel:
        """Update with optimistic locking. Raises if version mismatch."""
        current = self.read(key)
        if current is None:
            raise KeyError(f"Key {key!r} not found")
        if current.version != expected_version:
            raise ValueError(
                f"Version conflict on {key!r}: expected {expected_version}, got {current.version}"
            )
        # Bump version on successful update
        updated = entry.model_copy(update={"version": expected_version + 1})
        self._client.set(key, updated.model_dump_json())
        return updated

    def delete(self, key: str) -> None:
        """Remove an entry."""
        self._client.delete(key)

    def list_keys(self) -> list[str]:
        """List all keys (for discovery)."""
        keys = self._client.keys("*")
        # Upstash may return bytes — normalize
        return [k.decode("utf-8") if isinstance(k, bytes) else k for k in (keys or [])]

    def is_fresh(self, entry: BaseModel) -> bool:
        """Check if entry is within its TTL window."""
        age = time.time() - entry.timestamp
        return age < entry.ttl_seconds

    def snapshot(self) -> dict:
        """Return a JSON-serializable view of the entire store (for debugging)."""
        result = {}
        for key in self.list_keys():
            raw = self._client.get(key)
            if raw is None:
                continue
            if isinstance(raw, bytes):
                raw = raw.decode("utf-8")
            entry = self._deserialize(key, raw)
            if entry is not None:
                result[key] = entry.model_dump()
        return result

    def _deserialize(self, key: str, raw: str) -> BaseModel | None:
        """Reconstruct the right Pydantic model from a stored JSON string."""
        try:
            data = json.loads(raw)
        except (json.JSONDecodeError, TypeError):
            return None
        # Pick the schema based on key prefix
        if key.startswith("shipment:"):
            return ShipmentStatus.model_validate(data)
        if key.startswith("evt-"):
            return EventLog.model_validate(data)
        # Fallback: try both
        for model_cls in (ShipmentStatus, EventLog):
            try:
                return model_cls.model_validate(data)
            except Exception:
                continue
        return None


# Singleton — all agents share this one instance
memory = SharedEpistemicMemory(redis_client)
print("✓ SharedEpistemicMemory initialized (Upstash Redis backend)")


✓ SharedEpistemicMemory initialized (Upstash Redis backend)


## Cell 5 — LangChain Tools Wrapping the Memory Store

Agents never touch `memory` directly. They call **tools** that wrap memory
operations. This gives us:

1. **Schema enforcement** — tool inputs are typed, validated by Pydantic
2. **Auto-tracing** — every tool call is captured by MLflow
3. **Audit trail** — `source_agent_id` is injected automatically per tool

We define three tools:
- `log_event` — write a new event to memory (used by MonitoringAgent)
- `update_shipment_status` — write/update a shipment's status (used by LogisticsAgent)
- `read_memory` — read any entry by key (used by all agents for discovery)


In [7]:
from langchain.tools import tool
import uuid


@tool
def log_event(
    event_type: str,
    shipment_id: str,
    details: str,
    source_agent_id: str,
) -> str:
    """Log a discrete event to the shared memory.

    Args:
        event_type: One of 'disruption_detected', 'reroute_planned',
                    'customer_notified', 'delivery_confirmed'.
        shipment_id: The shipment this event relates to.
        details: Free-text description of what happened.
        source_agent_id: ID of the agent logging this event.

    Returns:
        The event_id of the newly created event.
    """
    event = EventLog(
        event_id=f"evt-{uuid.uuid4().hex[:8]}",
        event_type=event_type,
        shipment_id=shipment_id,
        details=details,
        source_agent_id=source_agent_id,
    )
    memory.write(event.event_id, event)
    return f"Logged event {event.event_id} ({event_type}) for shipment {shipment_id}"


@tool
def update_shipment_status(
    shipment_id: str,
    status: str,
    reason: str,
    source_agent_id: str,
) -> str:
    """Update a shipment's status in the shared memory.

    Args:
        shipment_id: The shipment to update.
        status: One of 'in_transit', 'delayed', 'rerouted', 'delivered', 'lost'.
        reason: Why the status changed.
        source_agent_id: ID of the agent making the update.

    Returns:
        Confirmation message with the new version number.
    """
    key = f"shipment:{shipment_id}"
    existing = memory.read(key)

    if existing is None:
        # First write — create new entry
        entry = ShipmentStatus(
            shipment_id=shipment_id,
            status=status,
            reason=reason,
            source_agent_id=source_agent_id,
        )
        memory.write(key, entry)
        return f"Created shipment {shipment_id} with status={status} (v{entry.version})"

    # Update existing — bump version
    updated = memory.update(
        key,
        existing.model_copy(update={"status": status, "reason": reason, "source_agent_id": source_agent_id}),
        expected_version=existing.version,
    )
    return f"Updated shipment {shipment_id} to status={status} (v{updated.version})"


@tool
def read_memory(key: str) -> str:
    """Read an entry from the shared memory by its key.

    Args:
        key: The memory key (e.g. 'shipment:SHP-001' or 'evt-abc12345').

    Returns:
        JSON string of the entry, or 'NOT_FOUND' if missing/stale.
    """
    entry = memory.read(key)
    if entry is None:
        return "NOT_FOUND"
    return entry.model_dump_json()


print("✓ Tools defined: log_event, update_shipment_status, read_memory")


✓ Tools defined: log_event, update_shipment_status, read_memory


## Cell 6 — Three Agents via `create_agent`

Now we wire up three agents, each with a focused role and a system prompt that
constrains its behavior. All three share the same `memory` singleton via the tools.

**MonitoringAgent** — watches for disruptions (storms, port closures, etc.) and
logs them as events. It does NOT touch shipment statuses directly.

**LogisticsAgent** — reads disruption events, decides on reroutes/delays, and
updates shipment statuses. It does NOT notify customers.

**CustomerNotificationAgent** — reads shipment statuses, identifies affected
customers, and logs notification events. It does NOT change shipment statuses.

This separation of concerns is what makes SEM powerful: each agent has a narrow
responsibility, and the shared memory is the coordination layer.


In [8]:
from langchain.agents import create_agent

# --- Monitoring Agent ---
monitoring_agent = create_agent(
    llm,
    tools=[log_event],
    system_prompt=(
        "You are the MonitoringAgent. Your job is to detect supply chain disruptions "
        "(storms, port closures, road blocks) and log them as events. "
        "Use the log_event tool with event_type='disruption_detected'. "
        "Always pass source_agent_id='monitoring-agent'. "
        "Do NOT update shipment statuses — that's the LogisticsAgent's job."
    ),
)

# --- Logistics Agent ---
logistics_agent = create_agent(
    llm,
    tools=[read_memory, update_shipment_status],
    system_prompt=(
        "You are the LogisticsAgent. Your job is to read disruption events from "
        "shared memory and update affected shipment statuses. "
        "First call read_memory to discover recent events, then call "
        "update_shipment_status with status='delayed' or 'rerouted'. "
        "Always pass source_agent_id='logistics-agent'. "
        "Do NOT notify customers — that's the CustomerNotificationAgent's job."
    ),
)

# --- Customer Notification Agent ---
customer_agent = create_agent(
    llm,
    tools=[read_memory, log_event],
    system_prompt=(
        "You are the CustomerNotificationAgent. Your job is to read shipment "
        "statuses from shared memory and log customer notification events for "
        "any shipment that is delayed, rerouted, or lost. "
        "First call read_memory to discover shipment statuses, then call "
        "log_event with event_type='customer_notified'. "
        "Always pass source_agent_id='customer-notification-agent'."
    ),
)

print("✓ Three agents created:")
print("  - MonitoringAgent (logs disruptions)")
print("  - LogisticsAgent (updates shipment statuses)")
print("  - CustomerNotificationAgent (logs notifications)")


✓ Three agents created:
  - MonitoringAgent (logs disruptions)
  - LogisticsAgent (updates shipment statuses)
  - CustomerNotificationAgent (logs notifications)


## Cell 7 — Run the End-to-End Workflow

We simulate a supply chain disruption:
1. A storm hits the route for shipment `SHP-001`
2. MonitoringAgent logs the disruption event
3. LogisticsAgent reads the event and marks the shipment as `delayed`
4. CustomerNotificationAgent reads the status and logs a notification event

Each agent invocation is auto-traced by MLflow. After running this cell, open
`http://localhost:5909` to see the full trace tree.


In [9]:
import json

def run_agent(agent, prompt: str, label: str):
    """Helper: invoke an agent and print the final response."""
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    result = agent.invoke(
        {"messages": [{"role": "user", "content": prompt}]},
    )
    final_msg = result["messages"][-1]
    print(f"\n{label} response:\n  {final_msg.content}")
    return result


# Step 1: MonitoringAgent detects a storm
run_agent(
    monitoring_agent,
    "A severe storm has closed the I-80 corridor in Nebraska. Shipment SHP-001 "
    "is currently in transit on this route. Log this disruption.",
    "MonitoringAgent",
)

# Step 2: LogisticsAgent reads the event and updates the shipment
run_agent(
    logistics_agent,
    "Check shared memory for recent disruption events. If any shipment is "
    "affected, update its status accordingly.",
    "LogisticsAgent",
)

# Step 3: CustomerNotificationAgent reads the status and notifies
run_agent(
    customer_agent,
    "Check shared memory for shipment statuses. For any shipment that is "
    "delayed, rerouted, or lost, log a customer_notified event.",
    "CustomerNotificationAgent",
)

# Final state of shared memory
print(f"\n{'='*60}")
print("  FINAL SHARED MEMORY STATE")
print(f"{'='*60}")
print(json.dumps(memory.snapshot(), indent=2, default=str))



  MonitoringAgent

MonitoringAgent response:
  The disruption has been logged.

  LogisticsAgent

LogisticsAgent response:
  

  CustomerNotificationAgent

CustomerNotificationAgent response:
  

  FINAL SHARED MEMORY STATE
{
  "evt-0aab95ad": {
    "event_id": "evt-0aab95ad",
    "event_type": "disruption_detected",
    "shipment_id": "SHP-001",
    "details": "Severe storm has closed the I-80 corridor in Nebraska, affecting shipment SHP-001 currently in transit on this route.",
    "source_agent_id": "monitoring-agent",
    "timestamp": 1782558948.497705,
    "ttl_seconds": 86400,
    "version": 1
  }
}


[Trace(trace_id=tr-ed216ef3063b2fd8909cdb2e26c6a45a), Trace(trace_id=tr-e5aebf70dd486ebb7304b9c7d863d390)]

## Cell 8 — Staleness Demo (TTL Expiry)

The FDS requires TTL/staleness tracking. Let's prove it works:
1. Write an entry with a very short TTL (2 seconds)
2. Read it immediately — should succeed
3. Wait 3 seconds
4. Read again — should return `None` (treated as missing)


In [10]:
# Write a short-lived entry
short_entry = EventLog(
    event_id="evt-stale-test",
    event_type="disruption_detected",
    shipment_id="SHP-002",
    details="Test entry with 2s TTL",
    source_agent_id="test",
    ttl_seconds=2,
)
memory.write("evt-stale-test", short_entry)

# Read immediately — should succeed
fresh = memory.read("evt-stale-test")
print(f"Immediate read: {'FOUND' if fresh else 'NOT_FOUND'} (expected: FOUND)")

# Wait for TTL to expire
print("Sleeping 3 seconds...")
time.sleep(3)

# Read again — should be None (stale)
stale = memory.read("evt-stale-test")
print(f"After 3s read:   {'FOUND' if stale else 'NOT_FOUND'} (expected: NOT_FOUND)")

print("\n✓ Staleness tracking works — stale entries are treated as missing.")


Immediate read: FOUND (expected: FOUND)
Sleeping 3 seconds...
After 3s read:   NOT_FOUND (expected: NOT_FOUND)

✓ Staleness tracking works — stale entries are treated as missing.


Trace(trace_id=tr-0db6784d3c06db20dd3ed66ed23e18b6)

## Findings & Next Steps

**What we proved:**
- ✅ Three agents can coordinate through a shared memory store (no direct agent-to-agent calls)
- ✅ Pydantic schemas enforce typed entries with `source_agent_id`, `version`, `ttl_seconds`
- ✅ Optimistic locking via `version` prevents silent overwrites
- ✅ TTL/staleness tracking works — stale entries are treated as missing
- ✅ MLflow auto-traces every LLM call, tool call, and agent invocation

**What we deferred (for the app phase):**
- ❌ Redis backend (currently in-memory `dict`)
- ❌ Pub/sub for real-time agent wake-ups (currently pull-based)
- ❌ Concurrency safety beyond optimistic locking (no Lua scripts yet)
- ❌ Persistence across restarts
- ❌ Error types: `SchemaValidationError`, `VersionConflictError`, `StaleEntryWarning`

**MLflow UI:** Open `http://localhost:5909` to inspect traces for this run.
Look for the experiment `supply-chain-disruption-poc`.

**Next step:** Once the pattern feels right, swap the in-memory store for
Redis (per FDS) and add pub/sub so agents can react to memory changes
in real time instead of polling.
